# The check on markets never looked at

Plan committed before these markets were downloaded: see [PLAN_more_inputs.md](PLAN_more_inputs.md) (*Clean checks*).

- **Markets:** Korea (KOSPI), Taiwan (TAIEX), Hong Kong (Hang Seng) and Japan (Nikkei 225). No choice in this project was made using them.
- **Same code and settings as step 9:** the four sets of inputs, seven models, test years 2017–2026, the same days for every set. Run once.
- **Expected:** the same pattern as step 9: no gain from B or C, and a close-to-close gain from D.

`more_inputs.py new_markets` runs the models. This notebook scores them with step 9's code.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import roc_auc_score

from classifiers import overview
from data import NEW_MARKETS, load_all
from evaluation import bootstrap_range, holm, mcnemar
from models import pooled
from more_inputs import MODELS, RUNS, SAVED, run_all

markets = load_all(include_holdout=True, markets=NEW_MARKETS)
preds = run_all(markets, folder=SAVED / "new_markets")  # loads the saved runs

def together(target, s, model):
    """One model's predictions on every market, one row per market-day."""
    return pd.concat([preds[target, s, model, m].assign(market=m) for m in markets])

def right(p):
    return (p["p_up"] > 0.5) == p["up"]

pd.Series({m: len(preds["close to close", "A", "LDA", m]) for m in markets}, name="test days").to_frame().T

,Korea,Taiwan,Hong Kong,Japan
test days,2396,2376,2361,2393


## Overview

Mean AUC over the 40 folds (4 markets × 10 test years), per model and set. A coin flip scores 0.50.

In [2]:
pd.DataFrame({f"{s}, {target}": overview({model: {m: preds[target, s, model, m] for m in markets} for model in MODELS})["mean AUC"]
              for target, sets in RUNS.items() for s in sets}).round(3)

,"A, close to close","B, close to close","C, close to close","D, close to close","A, open to close","D, open to close"
LDA,0.499,0.505,0.498,0.671,0.507,0.530
Logistic regression,0.506,0.511,0.499,0.679,0.512,0.536
KNN,0.506,0.514,0.508,0.657,0.505,0.522
SVM,0.495,0.508,0.504,0.673,0.506,0.523
Random forest,0.517,0.521,0.494,0.674,0.514,0.534
AdaBoost,0.505,0.502,0.503,0.674,0.508,0.525
Neural network,0.509,0.498,0.505,0.676,0.507,0.522


## Main result: better than A?

Each model with set B, C or D against the same model with set A, on the same days (all markets and years together). McNemar's test counts days where only one of the two was right. Holm adjusts over the 21 close-to-close comparisons. The AUC range is from whole weeks drawn at random.

In [3]:
def against_a(target, s, model):
    a, b = together(target, "A", model), together(target, s, model)
    assert a.index.equals(b.index) and (a["market"] == b["market"]).all()  # the same days
    only_b, only_a, p = mcnemar(b["up"], b["p_up"] > 0.5, a["p_up"] > 0.5)
    daily = pd.DataFrame({"up": b["up"], "a": a["p_up"], "b": b["p_up"]})
    auc_gain = lambda f: roc_auc_score(f["up"], f["b"]) - roc_auc_score(f["up"], f["a"])
    low, high = bootstrap_range(daily, auc_gain)
    return {"accuracy gain, points": 100 * (right(b).mean() - right(a).mean()), "AUC gain": auc_gain(daily),
            "AUC gain, low": low, "AUC gain, high": high,
            "days only it was right": only_b, "days only A was right": only_a, "p": p}

main = pd.DataFrame({(s, model): against_a("close to close", s, model) for s in "BCD" for model in MODELS}).T
main["p (Holm)"] = holm(main["p"])
main.round(3)

accuracy gain, points  AUC gain  AUC gain, low  \
B LDA                                 -1.018    -0.001         -0.012   
  Logistic regression                 -0.294     0.008         -0.001   
  KNN                                  0.000     0.008         -0.005   
  SVM                                 -0.325     0.011         -0.001   
  Random forest                        0.136     0.007         -0.003   
  AdaBoost                             0.346    -0.003         -0.014   
  Neural network                      -0.808    -0.011         -0.025   
C LDA                                 -0.777    -0.001         -0.017   
  Logistic regression                 -0.010     0.010         -0.004   
  KNN                                  0.168     0.003         -0.012   
  SVM                                 -0.031     0.001         -0.013   
  Random forest                       -0.903    -0.012         -0.027   
  AdaBoost                             1.312     0.017          0.001   
  Neural network                      -0.493    -0.001         -0.017   
D LDA                                 10.540     0.169          0.154   
  Logistic regression                 10.382     0.183          0.168   
  KNN                                  9.154     0.149          0.133   
  SVM                                 10.414     0.175          0.157   
  Random forest                       10.277     0.160          0.143   
  AdaBoost                            12.513     0.173          0.156   
  Neural network                      10.949     0.170          0.153   

                       AUC gain, high  days only it was right  \
B LDA                           0.010                  1137.0   
  Logistic regression           0.018                   513.0   
  KNN                           0.020                  1234.0   
  SVM                           0.022                   716.0   
  Random forest                 0.017                   972.0   
  AdaBoost                      0.008                  1584.0   
  Neural network                0.001                  1346.0   
C LDA                           0.015                  1687.0   
  Logistic regression           0.023                   727.0   
  KNN                           0.018                  1483.0   
  SVM                           0.014                   968.0   
  Random forest                 0.002                  1391.0   
  AdaBoost                      0.032                  2364.0   
  Neural network                0.014                  1814.0   
D LDA                           0.185                  2179.0   
  Logistic regression           0.199                  2221.0   
  KNN                           0.166                  2091.0   
  SVM                           0.191                  2387.0   
  Random forest                 0.177                  2387.0   
  AdaBoost                      0.191                  2828.0   
  Neural network                0.188                  2534.0   

                       days only A was right      p  p (Holm)  
B LDA                                 1234.0  0.049     0.681  
  Logistic regression                  541.0  0.406     1.000  
  KNN                                 1234.0  1.000     1.000  
  SVM                                  747.0  0.433     1.000  
  Random forest                        959.0  0.785     1.000  
  AdaBoost                            1551.0  0.568     1.000  
  Neural network                      1423.0  0.149     1.000  
C LDA                                 1761.0  0.214     1.000  
  Logistic regression                  728.0  1.000     1.000  
  KNN                                 1467.0  0.782     1.000  
  SVM                                  971.0  0.964     1.000  
  Random forest                       1477.0  0.112     1.000  
  AdaBoost                            2239.0  0.068     0.879  
  Neural network                      1861.0  0.448     1.000  
D LDA                    

Open to close (buy at the open, sell at the close): set D against A, Holm over these 7.

In [4]:
tradeable = pd.DataFrame({model: against_a("open to close", "D", model) for model in MODELS}).T
tradeable["p (Holm)"] = holm(tradeable["p"])
tradeable.round(3)

,"accuracy gain, points",AUC gain,"AUC gain, low","AUC gain, high",days only it was right,days only A was right,p,p (Holm)
LDA,1.723,0.024,0.016,0.032,1104.0,940.0,0.000,0.002
Logistic regression,2.196,0.020,0.011,0.029,986.0,777.0,0.000,0.000
KNN,0.914,0.019,0.008,0.030,1426.0,1339.0,0.102,0.306
SVM,0.315,0.011,0.001,0.021,1199.0,1169.0,0.551,0.837
Random forest,1.093,0.016,0.007,0.026,1304.0,1200.0,0.040,0.198
AdaBoost,1.219,0.022,0.009,0.035,1751.0,1635.0,0.048,0.198
Neural network,0.515,0.020,0.008,0.033,1783.0,1734.0,0.418,0.837


## Any edge at all?

Pooled accuracy minus the fair bar, per model and market. Holm adjusts within each set (28 tests), as in step 8.

In [5]:
edge = pd.DataFrame([{"target": target, "set": s, "model": model, "market": m,
                      "gain": pooled(preds[target, s, model, m])["accuracy %"] - pooled(preds[target, s, model, m])["fair bar %"],
                      "p": pooled(preds[target, s, model, m])["p vs fair"]}
                     for target, sets in RUNS.items() for s in sets for model in MODELS for m in markets])
edge["p (Holm)"] = edge.groupby(["target", "set"])["p"].transform(holm)
edge.groupby(["target", "set"]).agg(
    tests=("p", "size"), mean_gain=("gain", "mean"), above=("gain", lambda g: (g > 0).sum()),
    significant=("p", lambda p: (p < 0.05).sum()), significant_after_holm=("p (Holm)", lambda p: (p < 0.05).sum())).round(2)

tests  mean_gain  above  significant  \
target         set                                         
close to close A       28      -1.54      0           10   
               B       28      -1.82      0           13   
               C       28      -1.65      1           10   
               D       28       9.05     28           28   
open to close  A       28       0.75     18            2   
               D       28       1.89     21            8   

                    significant_after_holm  
target         set                          
close to close A                         3  
               B                         3  
               C                         0  
               D                        28  
open to close  A                         0  
               D                         7

Each set's gains over the fair bar, in points (model × market).

In [6]:
for (target, s), table in edge.groupby(["target", "set"]):
    print(f"set {s}, {target}")
    display(table.pivot(index="model", columns="market", values="gain").round(1))

set A, close to close


market,Hong Kong,Japan,Korea,Taiwan
model,,,,
AdaBoost,-1.6,-1.7,-5.8,-4.7
KNN,-1.4,-0.5,-2.2,-1.4
LDA,-0.6,-1.8,-2.3,-1.5
Logistic regression,-0.5,-0.0,-1.6,-1.1
Neural network,-1.0,-0.3,-2.3,-3.3
Random forest,-0.2,-1.4,-2.1,-0.2
SVM,-1.1,-0.3,-2.4,-0.1


set B, close to close


market,Hong Kong,Japan,Korea,Taiwan
model,,,,
AdaBoost,-1.1,-4.2,-2.1,-5.1
KNN,-1.6,-1.0,-1.6,-1.3
LDA,-0.3,-2.9,-3.7,-3.3
Logistic regression,-0.7,-0.6,-1.8,-1.3
Neural network,-0.1,-2.4,-3.6,-3.9
Random forest,-0.9,-0.6,-1.8,-0.1
SVM,-1.5,-1.1,-1.9,-0.8


set C, close to close


market,Hong Kong,Japan,Korea,Taiwan
model,,,,
AdaBoost,0.1,-3.0,-2.2,-3.4
KNN,-1.1,-1.7,-1.4,-0.6
LDA,-2.6,-1.5,-2.6,-2.5
Logistic regression,-1.6,-0.3,-0.8,-0.7
Neural network,-1.3,-1.8,-3.0,-2.7
Random forest,-2.1,-0.8,-2.6,-1.9
SVM,-1.5,-0.0,-1.1,-1.4


... (2 outputs left out)

set A, open to close


market,Hong Kong,Japan,Korea,Taiwan
model,,,,
AdaBoost,-0.4,1.7,-0.1,2.6
KNN,-0.2,0.8,0.0,1.9
LDA,-0.3,2.1,-1.0,0.6
Logistic regression,0.4,1.3,-0.1,0.0
Neural network,-0.9,1.5,-1.3,3.4
Random forest,0.6,0.9,0.5,3.4
SVM,1.0,1.0,0.4,1.6


set D, open to close


market,Hong Kong,Japan,Korea,Taiwan
model,,,,
AdaBoost,-1.1,2.3,1.3,6.0
KNN,-0.1,0.5,0.5,5.1
LDA,-0.2,1.1,2.3,4.9
Logistic regression,1.2,0.7,2.5,5.9
Neural network,-1.3,-0.6,1.4,5.1
Random forest,0.4,1.4,1.6,6.3
SVM,-1.0,-0.7,1.7,5.3


## 2017–2022 against 2023–2026

Accuracy gain over A in each period, in points, averaged over the seven models.

In [7]:
periods = {"2017-2022": range(2017, 2023), "2023-2026": range(2023, 2027)}
rows = {}
for target, sets in RUNS.items():
    for s in sets:
        if s == "A":
            continue
        for period, years in periods.items():
            gains = []
            for model in MODELS:
                a, b = together(target, "A", model), together(target, s, model)
                keep = a["year"].isin(years)
                gains.append(100 * (right(b)[keep].mean() - right(a)[keep].mean()))
            rows[f"{s}, {target}", period] = np.mean(gains)
pd.Series(rows).unstack().round(2)

,2017-2022,2023-2026
"B, close to close",-0.42,-0.05
"C, close to close",-0.25,0.12
"D, close to close",10.65,10.53
"D, open to close",0.85,1.59


## What tsfresh kept

For each market, the 11 training cutoffs (2015–2025): how often no feature passed (so the top 10 were used), and how many passed otherwise.

In [8]:
rows, kept = {}, Counter()
for m in markets:
    chosen = pd.read_pickle(SAVED / f"tsfresh_selection_close_to_close_{m}.pkl".replace(" ", "_"))
    passed = [len(cols) for cols, fell_back in chosen.values() if not fell_back]
    rows[m] = {"cutoffs": len(chosen), "fell back to the top 10": sum(f for _, f in chosen.values()),
               "features passing, median": np.median(passed) if passed else np.nan,
               "features passing, most": max(passed) if passed else np.nan}
    kept.update(c for cols, _ in chosen.values() for c in cols)
display(pd.DataFrame(rows).T)
pd.Series(dict(kept.most_common(10)), name="times kept").to_frame()

,cutoffs,fell back to the top 10,"features passing, median","features passing, most"
Korea,11.0,11.0,NaN,NaN
Taiwan,11.0,11.0,NaN,NaN
Hong Kong,11.0,11.0,NaN,NaN
Japan,11.0,11.0,NaN,NaN


,times kept
"ret__change_quantiles__f_agg_""var""__isabs_False__qh_0.8__ql_0.6",11
"ret__change_quantiles__f_agg_""var""__isabs_True__qh_0.8__ql_0.6",11
ret__symmetry_looking__r_0.05,11
"ret__fft_coefficient__attr_""imag""__coeff_7",11
ret__lempel_ziv_complexity__bins_2,10
"ret__cwt_coefficients__coeff_12__w_2__widths_(2, 5, 10, 20)",9
ret__partial_autocorrelation__lag_7,9
"ret__linear_trend__attr_""pvalue""",9
ret__energy_ratio_by_chunks__num_segments_10__segment_focus_1,9
ret__friedrich_coefficients__coeff_0__m_3__r_30,8


## Added after looking: is set D's gain a timing leak?

Step 9's check, unchanged. A move from after Asia's close should line up with the **overnight gap**, and with nothing after the open.

In [9]:
from data import load_other_moves, load_us
from features import make_other_markets

us, others = load_us(include_holdout=True), load_other_moves()
rows = {}
for m, df in markets.items():
    X = make_other_markets(df, us, others, m)
    real_open = df["Open"] != df["Close"].shift(1)
    gap = np.log(df["Open"] / df["Close"].shift(1)).where(real_open)
    after_open = np.log(df["Close"] / df["Open"]).where(real_open)
    for move in ["us_ret", "vix_move", "fx_move", "oil_move"]:
        rows[move, m] = {"with the overnight gap": X[move].corr(gap), "with day t after the open": X[move].corr(after_open),
                         "with day t+1 (should be about 0)": X[move].corr(df["ret"].shift(-1))}
pd.DataFrame(rows).T.round(2)

,,with the overnight gap,with day t after the open,with day t+1 (should be about 0)
us_ret,Korea,0.59,-0.06,-0.01
vix_move,Korea,-0.49,0.02,-0.02
fx_move,Korea,0.03,-0.02,-0.00
oil_move,Korea,0.10,0.04,0.01
us_ret,Taiwan,0.57,0.19,0.02
vix_move,Taiwan,-0.49,-0.18,-0.03
fx_move,Taiwan,-0.01,-0.02,0.01
oil_move,Taiwan,0.14,0.06,0.01
us_ret,Hong Kong,0.52,-0.08,0.01
vix_move,Hong Kong,-0.47,0.05,-0.04


## Added after looking: is the open partly stale?

Taiwan's US move lines up a little with the move after the open (0.19), and Taiwan is where set D wins open to close. If an index's first print still uses yesterday's prices for stocks that haven't traded yet, the move after the open carries on from the gap. Here is that for every market.

In [10]:
from data import MARKETS

rows = {}
for group, ms in [("new", markets), ("step 9", load_all(include_holdout=True, markets=MARKETS))]:
    for m, df in ms.items():
        gap = np.log(df["Open"] / df["Close"].shift(1))
        after_open = np.log(df["Close"] / df["Open"])
        rows[m] = {"markets": group, "gap with the move after the open": round(gap.corr(after_open), 2)}
pd.DataFrame(rows).T

,markets,gap with the move after the open
Korea,new,0.06
Taiwan,new,0.29
Hong Kong,new,0.03
Japan,new,0.21
Indonesia,step 9,0.11
Malaysia,step 9,-0.03
Philippines,step 9,0.09
Singapore,step 9,0.06
Thailand,step 9,-0.0
India,step 9,-0.04


## Conclusions

**The plan's expectation held: the same pattern as step 9.** B and C add nothing, and D wins close to close.

| Set against A (Holm over 21) | Accuracy, points | AUC | Significant after Holm |
|---|---|---|---|
| B: standard extras | −1.0 to +0.3 | −0.011 to +0.011 | 0 of 7 |
| C: tsfresh | −0.9 to +1.3 | −0.013 to +0.017 | 0 of 7 |
| D, close to close | **+9.2 to +12.5** | **+0.15 to +0.18** | **7 of 7** |
| D, open to close (Holm over 7) | +0.3 to +2.2 | +0.01 to +0.02 | 2 of 7 |

- **B and C:** no difference. Two models' AUC with B is 0.011 away from A's, just outside the planned ±0.01. tsfresh found nothing: in all 44 training cutoffs no feature passed, so every fold used the 10 least bad.
- **D, close to close:** all 28 model-market pairs beat the fair bar after Holm, by +4.9 to +7.5 points in Hong Kong, +6.8 to +8.4 in Korea, +8.9 to +10.5 in Taiwan and +10.5 to +12.7 in Japan. The gain is larger than in step 9 (+6.0 to +7.8). Why was not looked into.
- **A, B and C sit below the fair bar** (mean −1.5 to −1.8 points). These markets rose on 51.5–55.3% of days, so always guessing "up" is hard to beat. Six pairs are significantly below it after Holm, all AdaBoost or the neural network in Korea, Taiwan or Japan.
- **Old and new years agree:** D's gain is +10.7 points in 2017–2022 and +10.5 in 2023–2026. B and C are within ±0.5 in both.
- **Open to close:** D's edge over the fair bar is only in Taiwan (+4.9 to +6.3, all 7 models after Holm), where set A alone also scores up to +3.4 (not significant).

**Added after looking:**
- **Not a timing leak.** The US and VIX moves line up with the overnight gap (0.52 to 0.62, and −0.47 to −0.57), as they should, and not with day t+1 (0.04 or less).
- **Taiwan's open looks partly stale.** There, the move after the open carries on from the gap (0.29), as in Japan (0.21). In the other eight markets this is −0.04 to +0.11. So Taiwan's open-to-close edge most likely comes from a lagging opening number, as with the Philippines and Malaysia in step 5b, not from a trade you could make.

**Data caveats:** the fixed bad-print rule blanked 148 Taiwan dollar closes (Yahoo's 2011–2016 series is broken: it jumps about 2.5% back and forth) and 44 Hong Kong dollar closes (the dollar is pegged, so the rule's "typical move" is tiny and ordinary closes get blanked). Those days drop out of all four sets alike.